In [100]:
import pandas as pd

In [101]:
df = pd.read_csv('../data/Titanic-Dataset.csv')

In [102]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [103]:
df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace=True)

### 01. Handling Missing Values and Creatig a Missing Indicator Feature

In [104]:
# Missing Indicator
for col in ['Age', 'Embarked']:
    df['Missing'+col] = df[col].isnull().astype(int)

In [105]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,MissingAge,MissingEmbarked
0,0,3,male,22.0,1,0,7.2500,S,0,0
1,1,1,female,38.0,1,0,71.2833,C,0,0
2,1,3,female,26.0,0,0,7.9250,S,0,0
3,1,1,female,35.0,1,0,53.1000,S,0,0
4,0,3,male,35.0,0,0,8.0500,S,0,0


In [106]:
# Mean Imputation on Age
df['Age'] = df['Age'].fillna(df['Age'].mean())

# Mode Imputation on Embarked
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

df.isna().sum()

Survived           0
Pclass             0
Sex                0
Age                0
SibSp              0
Parch              0
Fare               0
Embarked           0
MissingAge         0
MissingEmbarked    0
dtype: int64

### 02. Splitting 

In [107]:
from sklearn.model_selection import train_test_split

X = df.drop("Survived", axis=1)
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 02. Encoding 'Sex' and 'Embarked'

In [108]:
# One hot encoding on Embarked and Sex
X_train_encoded = pd.get_dummies(X_train, columns=['Embarked', 'Sex'])
X_test_encoded = pd.get_dummies(X_test, columns=['Embarked', 'Sex'])
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)
X_train_encoded.shape

(712, 12)

In [109]:
X_train_encoded.info()

<class 'pandas.DataFrame'>
Index: 712 entries, 331 to 102
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Pclass           712 non-null    int64  
 1   Age              712 non-null    float64
 2   SibSp            712 non-null    int64  
 3   Parch            712 non-null    int64  
 4   Fare             712 non-null    float64
 5   MissingAge       712 non-null    int64  
 6   MissingEmbarked  712 non-null    int64  
 7   Embarked_C       712 non-null    bool   
 8   Embarked_Q       712 non-null    bool   
 9   Embarked_S       712 non-null    bool   
 10  Sex_female       712 non-null    bool   
 11  Sex_male         712 non-null    bool   
dtypes: bool(5), float64(2), int64(5)
memory usage: 48.0 KB


### 03. Initial Training without Scaling

In [110]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = LogisticRegression(max_iter=1000)
model.fit(X_train_encoded, y_train)

y_pred = model.predict(X_test_encoded)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8156424581005587

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.87      0.85       105
           1       0.80      0.74      0.77        74

    accuracy                           0.82       179
   macro avg       0.81      0.80      0.81       179
weighted avg       0.81      0.82      0.81       179


Confusion Matrix:
 [[91 14]
 [19 55]]


### 04. Trying Standard Scalar

In [121]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_encoded[['Age', 'Fare']] = scaler.fit_transform(X_train[['Age', 'Fare']])
X_test_encoded[['Age', 'Fare']] = scaler.transform(X_test[['Age', 'Fare']])

In [122]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = LogisticRegression(max_iter=1000)
model.fit(X_train_encoded, y_train)

y_pred = model.predict(X_test_encoded)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8212290502793296

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.87      0.85       105
           1       0.80      0.76      0.78        74

    accuracy                           0.82       179
   macro avg       0.82      0.81      0.81       179
weighted avg       0.82      0.82      0.82       179


Confusion Matrix:
 [[91 14]
 [18 56]]


### 05. Trying MinMax Scalar

In [119]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_encoded[['Age', 'Fare']] = scaler.fit_transform(X_train[['Age', 'Fare']])
X_test_encoded[['Age', 'Fare']] = scaler.transform(X_test[['Age', 'Fare']])

In [120]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = LogisticRegression(max_iter=1000)
model.fit(X_train_encoded, y_train)

y_pred = model.predict(X_test_encoded)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8044692737430168

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.85      0.84       105
           1       0.77      0.74      0.76        74

    accuracy                           0.80       179
   macro avg       0.80      0.80      0.80       179
weighted avg       0.80      0.80      0.80       179


Confusion Matrix:
 [[89 16]
 [19 55]]


### 06. Trying Robust Scalar

In [125]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

X_train_encoded[['Age', 'Fare']] = scaler.fit_transform(X_train[['Age', 'Fare']])
X_test_encoded[['Age', 'Fare']] = scaler.transform(X_test[['Age', 'Fare']])

In [126]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = LogisticRegression(max_iter=1000)
model.fit(X_train_encoded, y_train)

y_pred = model.predict(X_test_encoded)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8156424581005587

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.87      0.85       105
           1       0.80      0.74      0.77        74

    accuracy                           0.82       179
   macro avg       0.81      0.80      0.81       179
weighted avg       0.81      0.82      0.81       179


Confusion Matrix:
 [[91 14]
 [19 55]]
